In [1]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


# ==========================================
# 2. LOAD DATA
# ==========================================
df = pd.read_csv("Data.csv")


# ==========================================
# 3. REMOVE UNNECESSARY COLUMNS
# ==========================================

# Drop instant FIRST (as you wanted ✅)
df = df.drop('instant', axis=1)

# Convert date
df['dteday'] = pd.to_datetime(df['dteday'])

# Feature Engineering
df['year'] = df['dteday'].dt.year
df['month'] = df['dteday'].dt.month
df['day'] = df['dteday'].dt.day
df['dayofweek'] = df['dteday'].dt.dayofweek

# Drop leakage columns + date
df = df.drop(['dteday', 'casual', 'registered'], axis=1)


# ==========================================
# 4. DEFINE FEATURES & TARGET
# ==========================================
X = df.drop('cnt', axis=1)
y = df['cnt']


# ==========================================
# 5. TRAIN-TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ==========================================
# 6. MODELS
# ==========================================
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVR())
    ])
}


# ==========================================
# 7. BASE MODEL TRAINING
# ==========================================
results = {}

print("\n=== Base Models ===\n")

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results[name] = {"RMSE": rmse, "R2": r2}

    print(f"{name}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2: {r2:.4f}")
    print("-" * 40)


# ==========================================
# 8. GRID SEARCH (TUNING)
# ==========================================

print("\n=== GridSearch Tuning ===\n")

# Random Forest
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)


# Gradient Boosting
gb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1]
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_params,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
gb_grid.fit(X_train, y_train)


# SVM
svm_params = {
    'svm__C': [1, 10],
    'svm__gamma': ['scale']
}

svm_grid = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVR())
    ]),
    svm_params,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
svm_grid.fit(X_train, y_train)


# ==========================================
# 9. COLLECT ALL MODELS
# ==========================================
all_models = {}

# Base models
for name, model in models.items():
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    all_models[name] = {"model": model, "RMSE": rmse, "R2": r2}

# Tuned models
tuned_models = {
    "Random Forest (Tuned)": rf_grid.best_estimator_,
    "Gradient Boosting (Tuned)": gb_grid.best_estimator_,
    "SVM (Tuned)": svm_grid.best_estimator_
}

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    all_models[name] = {"model": model, "RMSE": rmse, "R2": r2}


# ==========================================
# 10. SELECT BEST MODEL
# ==========================================
best_model_name = min(all_models, key=lambda x: all_models[x]['RMSE'])
best_model = all_models[best_model_name]['model']

print("\n🏆 BEST MODEL:", best_model_name)
print("Performance:", all_models[best_model_name])


# ==========================================
# 11. SAVE MODEL
# ==========================================
joblib.dump(best_model, "best_bike_model.pkl")

print("\n✅ Model saved successfully!")


=== Base Models ===

Linear Regression
RMSE: 825.99
R2: 0.8299
----------------------------------------
Decision Tree
RMSE: 886.08
R2: 0.8042
----------------------------------------
Random Forest
RMSE: 685.28
R2: 0.8829
----------------------------------------
Gradient Boosting
RMSE: 663.17
R2: 0.8903
----------------------------------------
SVM
RMSE: 1999.23
R2: 0.0032
----------------------------------------

=== GridSearch Tuning ===


🏆 BEST MODEL: Gradient Boosting (Tuned)
Performance: {'model': GradientBoostingRegressor(learning_rate=0.05, n_estimators=200, random_state=42), 'RMSE': 653.8872997736651, 'R2': 0.8933712837350298}

✅ Model saved successfully!


In [2]:
df

,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt,year,month,day,dayofweek
0,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,985,2011,1,1,5
1,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,801,2011,1,2,6
2,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,1349,2011,1,3,0
3,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,1562,2011,1,4,1
4,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,1600,2011,1,5,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
726,1,1,12,0,4,1,2,0.254167,0.226642,0.652917,0.350133,2114,2012,12,27,3
727,1,1,12,0,5,1,2,0.253333,0.255046,0.590000,0.155471,3095,2012,12,28,4
728,1,1,12,0,6,0,2,0.253333,0.242400,0.752917,0.124383,1341,2012,12,29,5
729,1,1,12,0,0,0,1,0.255833,0.231700,0.483333,0.350754,1796,2012,12,30,6


In [3]:
df["holiday"].unique()

array([0, 1], dtype=int64)

In [4]:
model = joblib.load("best_bike_model.pkl")

In [7]:
X_test.iloc[0]

season           4.000000
yr               1.000000
mnth            12.000000
holiday          0.000000
weekday          2.000000
workingday       1.000000
weathersit       1.000000
temp             0.475833
atemp            0.469054
hum              0.733750
windspeed        0.174129
year          2012.000000
month           12.000000
day              4.000000
dayofweek        1.000000
Name: 703, dtype: float64

In [12]:
y_test.iloc[0]

6606

In [20]:
model.predict(X_test.iloc[0].values.reshape(1, -1))

c:\Users\Dell\anaconda3\envs\DL\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(


array([6552.03258786])

In [21]:
X_test.iloc[0].values.reshape(1, -1)

array([[4.00000e+00, 1.00000e+00, 1.20000e+01, 0.00000e+00, 2.00000e+00,
        1.00000e+00, 1.00000e+00, 4.75833e-01, 4.69054e-01, 7.33750e-01,
        1.74129e-01, 2.01200e+03, 1.20000e+01, 4.00000e+00, 1.00000e+00]])

In [22]:
model.predict(X_test.iloc[0:5])

array([6552.03258786, 1575.85739811, 3371.63895741, 4312.26680549,
       7274.7033816 ])

In [24]:
model.predict(X_test.iloc[:1])

array([6552.03258786])

In [25]:
X_test.iloc[:1]

,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,year,month,day,dayofweek
703,4,1,12,0,2,1,1,0.475833,0.469054,0.73375,0.174129,2012,12,4,1


In [26]:
test_data = {
  "season": 4,
  "yr": 1,
  "mnth": 12,
  "holiday": 0,
  "weekday": 2,
  "workingday": 1,
  "weathersit": 1,
  "temp": 0.475833,
  "atemp": 0.469054,
  "hum": 0.73375,
  "windspeed": 0.174129,
  "year": 2012,
  "month": 12,
  "day": 4,
  "dayofweek": 1
}

In [27]:
test_data

{'season': 4,
 'yr': 1,
 'mnth': 12,
 'holiday': 0,
 'weekday': 2,
 'workingday': 1,
 'weathersit': 1,
 'temp': 0.475833,
 'atemp': 0.469054,
 'hum': 0.73375,
 'windspeed': 0.174129,
 'year': 2012,
 'month': 12,
 'day': 4,
 'dayofweek': 1}

In [28]:
test_data = pd.DataFrame([test_data])

In [29]:
test_data

,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,year,month,day,dayofweek
0,4,1,12,0,2,1,1,0.475833,0.469054,0.73375,0.174129,2012,12,4,1


In [31]:
model.predict(test_data)[0]

6552.03258785918